
# 01 — SEC Financial Data Download and Extraction

## 1. Research Objective

This project investigates the relationship between financial leverage and profitability among publicly traded U.S. manufacturing firms.

Motivated by Dalci (2018), *Impact of financial leverage on profitability of listed manufacturing firms in China*, we examine whether the leverage–profitability relationship in the United States is approximately linear, quadratic (potentially inverted-U-shaped), or more generally nonlinear.

Our primary research question is:

> What is the functional relationship between balance-sheet leverage and profitability among publicly traded U.S. manufacturing firms?

The project emphasizes **statistical learning, functional estimation, and predictive model comparison**, rather than causal identification.

Subsequent analysis in R will consider linear and polynomial regression, regularization, regression splines, smoothing splines, generalized additive models (GAMs), bootstrap uncertainty, and cross-validation.

This notebook is restricted to **data acquisition, accounting-variable construction, and validation**. No statistical modeling or visualization is performed here.

---

## 2. Data Source and Sample Definition

### Data source

Financial statement information is obtained from the U.S. Securities and Exchange Commission (SEC), primarily through the EDGAR XBRL Company Facts API:

- SEC Company Facts: https://data.sec.gov/api/xbrl/companyfacts/
- SEC EDGAR: https://www.sec.gov/edgar

Company Facts provides standardized financial observations, including their accounting taxonomy, reporting period, filing date, and filing accession number.

### Target population

The primary candidate universe consists of companies satisfying the following criteria:

1. Classified as manufacturing under **SIC codes 2000–3999**.
2. Listed on a selected U.S. securities exchange: Nasdaq, NYSE, CBOE, or TXSE.
3. Incorporated in a recognized U.S. state or the District of Columbia.
4. Identifiable by a unique SEC Central Index Key (CIK).

The CIK is used as the primary company identifier, preventing duplicate observations for companies with multiple ticker symbols.

### Sample selection

The preliminary feasibility investigation (`00_data_feasibility.ipynb`) produced the following counts:

| Screening stage | Firms |
|---|---:|
| Manufacturing SIC candidates | 2,407 |
| Listed on selected U.S. exchanges | 2,073 |
| Confirmed U.S. incorporation and selected exchange listing | 1,467 |

Among the 2,073 exchange-listed manufacturing candidates, 427 had populated non-U.S. incorporation codes and 179 had missing incorporation metadata.

The 179 firms with unknown incorporation status are excluded from the primary candidate universe rather than automatically classified as foreign firms.

**The 1,467 firms constitute the candidate universe, not the final analytical sample.** The final sample will depend on the availability and validity of fiscal-year financial observations.

The universe is derived from currently available SEC metadata. Consequently, it may exclude companies that were publicly traded in 2024 but are no longer represented in the current ticker universe.

---

## 3. Financial Reporting Period

The analysis uses a **single fiscal-year 2024 cross section**.

For each company, the pipeline identifies an eligible annual filing and extracts accounting observations from the same filing accession and fiscal-year-end date.

The default selection rules are:

- Filing form: `10-K`
- Filing fiscal year (`fy`): 2024
- Fiscal period (`fp`): `FY`
- Fiscal-year-end date: within calendar year 2024
- Balance-sheet observations: point-in-time values
- Income-statement observations: approximately annual durations (330–380 days)

Balance-sheet variables are measured at fiscal year-end, whereas net income and revenue are measured over the corresponding annual reporting period.

The pipeline retains the selected filing accession, fiscal-end date, filing date, and original accounting tags for auditing.

---

## 4. Variable Definitions

### 4.1 Profitability: Return on Assets

Our primary profitability measure is return on assets (ROA):

$$
\operatorname{ROA}_i
=
\frac{\operatorname{NetIncome}_i}
{\operatorname{TotalAssets}_i}.
$$

The numerator is annual net income, and the denominator is total assets at the selected fiscal year-end.

Candidate US-GAAP tags include:

- `NetIncomeLoss`
- `ProfitLoss`
- `Assets`

Negative ROA values are retained because losses may exceed year-end assets, particularly among smaller or early-stage firms.

### 4.2 Financial Leverage: Liabilities-to-Assets Ratio

Our primary leverage measure is:

$$
\operatorname{Leverage}_i
=
\frac{\operatorname{TotalLiabilities}_i}
{\operatorname{TotalAssets}_i}.
$$

This measures **balance-sheet leverage**, not exclusively interest-bearing borrowing.

Total liabilities include obligations such as accounts payable, accrued expenses, lease liabilities, and financial debt.

The feasibility investigation found that this measure offers substantially greater coverage than the available interest-bearing debt concepts.

A liabilities-to-assets ratio exceeding one is possible when a company has negative book equity. Such observations are not automatically excluded.

### 4.3 Additional Accounting Variables

We retain the following variables for potential controls, alternative specifications, and data validation:

- Total assets
- Total liabilities
- Stockholders' equity
- Consolidated total equity
- Noncontrolling interests, where reported
- Annual revenue
- Net income

Potential subsequent analyses may also consider firm size, industry classifications, and alternative profitability or leverage measures.

These variables are retained in their original USD amounts. Any transformations used for statistical modeling will be documented separately in R.

---

## 5. Balance-Sheet Identity and Missing Liabilities

The accounting identity is:

$$
A_i=L_i+E_i,
$$

where $A_i$ denotes assets, $L_i$ denotes liabilities, and $E_i$ denotes consolidated total equity.

When total liabilities are unavailable, we may reconstruct them as:

$$
\widehat{L}_i=A_i-E_i.
$$

The pipeline follows a conservative hierarchy:

1. **Reported:** Use the directly reported `Liabilities` value whenever available.
2. **Reconstructed:** If liabilities are missing, use assets minus consolidated total equity when a suitable equity measure is available.
3. **Provisional:** If only stockholders' equity is available and noncontrolling interests cannot be established, do not automatically accept the reconstruction.
4. **Missing:** Retain missing values when neither direct reporting nor an acceptable reconstruction is available.

Directly reported liabilities are never overwritten by implied values.

The pipeline records the source of each liabilities observation and computes accounting-identity discrepancies for validation.

### Feasibility results

In the 30-company pilot:

| Measure | Firms |
|---|---:|
| Available FY2024 assets and net income | 26 |
| Directly reported liabilities | 24 |
| Additional reconstructed liabilities | 1 |
| Final usable ROA and leverage pairs | 25 |

Among the 24 firms with directly reported liabilities, 23 satisfied the tested balance-sheet identity within 0.1% of assets.

These results motivated the finalized extraction rules.

---

## 6. Data Acquisition and Reproducibility

The reusable pipeline is implemented in:

`code/pipeline/data_download.py`

The notebook calls this module rather than duplicating download and extraction logic.

The pipeline is designed to be **parameterized, auditable, and resumable**.

### Configurable parameters

The `DownloadConfig` object controls:

- Target fiscal year
- Accepted filing forms
- Annual reporting-period duration
- Fiscal-year-end selection policy
- SEC request headers and research contact
- Request delays, timeouts, and retries
- Raw-data cache and processed-output directories
- Accounting tag preferences
- Accounting-identity validation tolerance
- Output saving and cache reuse

### Caching strategy

Each SEC Company Facts response is stored as a separate JSON file indexed by its ten-digit CIK:

`data/raw/companyfacts/CIK##########.json`

Existing valid cached responses are reused on subsequent runs.

This separates **raw data acquisition** from **financial-variable construction**, allowing the extraction logic to be modified without repeating the full download.

### Output datasets

The pipeline generates:

- `data/processed/manufacturing_financials_2024_full.csv` — all eligible candidate firms, including missing observations and audit fields.
- `data/processed/manufacturing_financials_2024_analysis.csv` — firms with nonmissing ROA and leverage.
- `data/processed/sec_download_log_2024.csv` — download status and errors.

The complete dataset preserves CIKs, reporting dates, accession numbers, selected accounting tags, liabilities provenance, and accounting-identity diagnostics.

The analysis-ready CSV will be imported into R for subsequent statistical work.

---

## 7. Methodological Limitations

### 7.1 Fiscal-Year Selection

The default extraction requires the selected FY2024 fiscal-year-end date to fall within calendar year 2024.

Although consistent with the feasibility investigation, this is not a universal definition of fiscal year 2024.

Some companies use reporting calendars that do not align neatly with calendar years, and the SEC's `fy` field refers to the fiscal year associated with a filing rather than necessarily identifying the period of every individual fact.

Our strict selection rule may therefore exclude otherwise relevant companies or observations.

The fiscal-end selection policy is configurable and should be reconsidered in sensitivity analyses.

### 7.2 Accounting Comparability

Different companies may report economically related quantities under different US-GAAP tags.

For example:

- `NetIncomeLoss` and `ProfitLoss` may differ in their treatment of noncontrolling interests.
- Revenue tags may reflect different reporting conventions.
- Financial statement concepts can change across accounting taxonomies and reporting periods.

The extraction pipeline uses a predefined tag hierarchy, but tag availability does not guarantee complete economic comparability.

We therefore preserve the original accounting tag associated with each extracted observation.

Before statistical modeling, the final dataset should be audited for inconsistent definitions, implausible values, and unusual reporting conventions.

### 7.3 Reconstructed Liabilities

Reconstructing liabilities from assets and equity requires a consistent definition of total equity.

Stockholders' equity may exclude noncontrolling interests, whereas consolidated total equity includes them.

Using an incomplete equity measure could systematically overstate implied liabilities.

Accordingly, the pipeline distinguishes directly reported, reconstructed, and provisional observations.

Reconstructed values should be validated against original financial statements for a sample of firms. Subsequent statistical analyses should also assess sensitivity to excluding reconstructed observations.

---

## 8. Additional Considerations for Statistical Analysis

Several issues will be addressed in the R analysis rather than during data acquisition.

**Extreme observations.** ROA may be highly negative for loss-making firms, and leverage may exceed one. These values are not automatically errors. Robustness analyses should assess their influence without arbitrarily deleting observations.

**Shared denominators.** Both ROA and leverage use total assets in the denominator. This may introduce mechanical statistical dependence and complicate interpretation.

**Sample selection.** Firms with unavailable financial data, missing incorporation metadata, or no current ticker record may differ systematically from included firms.

**Association versus causation.** The cross-sectional relationship between leverage and profitability cannot establish the causal effect of financing decisions on firm performance. Reverse causality, omitted variables, and firm heterogeneity remain possible.

**Generalizability.** The analysis concerns the selected SEC-reporting, U.S.-incorporated manufacturing universe and should not be interpreted as representative of all U.S. manufacturing businesses.

---

## 9. Execution

The following cells will:

1. Import the reusable SEC data pipeline.
2. Load the previously screened eligible-firm universe.
3. Specify the download and extraction configuration.
4. Execute the resumable Company Facts download.
5. Construct and validate the FY2024 financial dataset.
6. Report extraction coverage, missingness, and accounting-identity diagnostics.
7. Export the resulting datasets for analysis in R.

No statistical models are estimated in this notebook.

---

## Reference

Dalci, I. (2018). Impact of financial leverage on profitability of listed manufacturing firms in China. *Pacific Accounting Review*. https://doi.org/10.1108/PAR-01-2018-0008


In [1]:
import sys
from pathlib import Path

# Locate the project root regardless of notebook working directory.
current = Path.cwd().resolve()

PROJECT_ROOT = next(
    p for p in (current, *current.parents)
    if (p / "code" / "pipeline" / "data_download.py").is_file()
)

CODE_DIR = PROJECT_ROOT / "code"

# Make code/pipeline importable.
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from pipeline.data_download import DownloadConfig, run_pipeline

print(f"Project root: {PROJECT_ROOT}")
print("Pipeline imported successfully.")

Project root: /Users/jonathanma/Desktop/Projects/lev-v-prof
Pipeline imported successfully.


In [2]:
config = DownloadConfig(
    fiscal_year=2024,
    sic_min=2000,
    sic_max=3999,
    exchanges=("Nasdaq", "NYSE", "CBOE", "TXSE"),
    require_us_incorporation=True,
    include_unknown_incorporation=False,
    user_agent="PvL Project rucstudentfinanceassociation@gmail.com",
    max_firms=None,  # Full eligible manufacturing universe
)

results = run_pipeline(config)

SEC submissions metadata:   0%|          | 0/7988 [00:00<?, ?it/s]

SEC Company Facts:   0%|          | 0/1467 [00:00<?, ?it/s]

Extracting financials:   0%|          | 0/1467 [00:00<?, ?it/s]

In [3]:
f = results["financials"]

print(f"Total firms: {len(f):,}")
print(f"Usable firms: {f['has_core_variables'].sum():,}")
print(f"Coverage: {f['has_core_variables'].mean():.1%}")

print("\nLiabilities sources:")
print(f["liabilities_source"].value_counts(dropna=False))

print("\nAccounting review flags:")
print(f["accounting_review_flag"].sum())

Total firms: 1,467
Usable firms: 1,285
Coverage: 87.6%

Liabilities sources:
liabilities_source
reported                             1153
missing                               114
reconstructed_consolidated_equity      71
current_plus_noncurrent                66
provisional_equity_only                63
Name: count, dtype: int64

Accounting review flags:
53


In [6]:
a = results["financials"].loc[
    results["financials"]["has_core_variables"]
].copy()

print("Primary sample:", len(a))

clean = a.loc[
    ~a["accounting_review_flag"].fillna(False)
]

print("Accounting-clean sample:", len(clean))

print("\nPrimary sample quantiles:")
display(a[["roa", "leverage"]].quantile([.01, .5, .99]))

print("\nAccounting-clean sample quantiles:")
display(clean[["roa", "leverage"]].quantile([.01, .5, .99]))

Primary sample: 1285
Accounting-clean sample: 1232

Primary sample quantiles:


,roa,leverage
0.01,-5.012585,0.039366
0.50,-0.059925,0.473934
0.99,0.305808,3.098436



Accounting-clean sample quantiles:


,roa,leverage
0.01,-5.213246,0.040166
0.50,-0.064541,0.465694
0.99,0.314613,3.050322



## 5. Final Sample Construction and Data Quality

This notebook constructs a cross-sectional dataset of U.S.-incorporated, publicly listed manufacturing firms using SEC EDGAR Company Facts. Manufacturing firms are identified using SIC codes 2000–3999, with fiscal year 2024 as the reference period.

The two primary variables are:

$$
\text{Leverage}_i
=
\frac{\text{Total Liabilities}_i}{\text{Total Assets}_i},
\qquad
\text{ROA}_i
=
\frac{\text{Net Income}_i}{\text{Total Assets}_i}.
$$

Annual financial values are selected using filing-level consistency checks to avoid mixing incompatible reporting periods. Liabilities are obtained from directly reported values where possible, followed by validated component-based or consolidated-equity reconstruction.

### Sample Construction

| Category | Firms |
|---|---:|
| Eligible manufacturing firms | 1,467 |
| Primary analytical sample | 1,285 |
| Coverage | 87.6% |
| Excluded from primary sample | 182 |
| Accounting review flags | 53 |
| Accounting-clean analytical sample | 1,232 |

The primary sample requires both ROA and leverage to be available. Provisional leverage values derived from stockholders' equity alone are not admitted into the primary sample.

### Liabilities Provenance

| Source | Firms | Usable |
|---|---:|---:|
| Directly reported liabilities | 1,153 | 1,148 |
| Current + noncurrent liabilities | 66 | 66 |
| Reconstructed from consolidated equity | 71 | 71 |
| Provisional equity-based estimate | 63 | 0 |
| Missing | 114 | 0 |

The 53 accounting review flags occur among firms with directly reported liabilities. These observations are retained in the primary sample but identified for sensitivity analysis.

### Distributional Diagnostics

The financial ratios exhibit substantial right-tail behavior in leverage and a pronounced negative tail in ROA. Of the 1,285 usable observations, 163 satisfy at least one exploratory extreme-value criterion: absolute ROA greater than 1 or leverage greater than 2.

These observations are not automatically treated as errors. Small asset denominators, negative book equity, and differences in business models may generate economically meaningful extreme ratios.

Excluding the 53 accounting-flagged observations produces similar ROA and leverage quantiles. Nevertheless, the accounting flags remain relevant to subsequent sensitivity analyses.

No winsorization, trimming, or statistical transformations are applied during data extraction.


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Define output directory
# --------------------------------------------------

PROCESSED = Path("data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 2. Construct analysis samples
# --------------------------------------------------

financials = results["financials"].copy()

financials["accounting_review_flag"] = (
    financials["accounting_review_flag"]
    .fillna(False)
    .astype(bool)
)

financials["has_core_variables"] = (
    financials["has_core_variables"]
    .fillna(False)
    .astype(bool)
)

primary = financials.loc[
    financials["has_core_variables"]
].copy()

accounting_clean = primary.loc[
    ~primary["accounting_review_flag"]
].copy()

# --------------------------------------------------
# 3. Validate sample consistency
# --------------------------------------------------

assert len(financials) == 1467
assert len(primary) == 1285
assert len(accounting_clean) == 1232

assert primary[["roa", "leverage"]].notna().all().all()

assert np.isfinite(
    primary[["roa", "leverage"]].to_numpy(dtype=float)
).all()

assert (primary["assets"] > 0).all()

assert not primary["liabilities_source"].isin(
    ["missing", "provisional_equity_only"]
).any()

# --------------------------------------------------
# 4. Export datasets
# --------------------------------------------------

financials.to_csv(
    PROCESSED / "financials_full_2024.csv",
    index=False
)

primary.to_csv(
    PROCESSED / "financials_primary_2024.csv",
    index=False
)

accounting_clean.to_csv(
    PROCESSED / "financials_accounting_clean_2024.csv",
    index=False
)

# --------------------------------------------------
# 5. Export summary diagnostics
# --------------------------------------------------

summary = pd.DataFrame({
    "metric": [
        "eligible_firms",
        "primary_sample",
        "accounting_clean_sample",
        "primary_coverage",
        "accounting_review_flags",
        "extreme_observations"
    ],
    "value": [
        len(financials),
        len(primary),
        len(accounting_clean),
        len(primary) / len(financials),
        int(financials["accounting_review_flag"].sum()),
        int((
            (primary["roa"].abs() > 1)
            | (primary["leverage"] > 2)
        ).sum())
    ]
})

summary.to_csv(
    PROCESSED / "sample_summary_2024.csv",
    index=False
)

print("Final extraction complete.")
print(f"Eligible firms: {len(financials):,}")
print(f"Primary sample: {len(primary):,}")
print(f"Accounting-clean sample: {len(accounting_clean):,}")
print(f"Coverage: {len(primary) / len(financials):.1%}")

print("\nExported files:")
for path in sorted(PROCESSED.glob("*2024.csv")):
    print(f"  {path}")



## 6. Statistical Considerations

The extracted data form a single-year cross-section rather than a longitudinal panel. Accordingly, the subsequent analysis focuses on associations between leverage and profitability, not causal effects.

The baseline specification is

$$
\text{ROA}_i
=
\beta_0+\beta_1\text{Leverage}_i+\varepsilon_i.
$$

The central research question is whether the relationship is better represented by a nonlinear function:

$$
\text{ROA}_i
=
f(\text{Leverage}_i)+\varepsilon_i.
$$

Three features of the extracted sample motivate additional modeling considerations.

**1. Extreme observations.** Both financial ratios exhibit heavy tails, particularly among firms with small asset bases. These observations may have substantial influence on squared-error-based estimators and predictive performance metrics.

**2. Industry heterogeneity.** SIC 2000–3999 includes substantially different manufacturing industries, ranging from pharmaceuticals to machinery and food production. Industry composition may confound the estimated leverage–profitability relationship. Industry-adjusted specifications will therefore be considered.

**3. Accounting quality.** Fifty-three primary-sample observations have accounting reconciliation flags. The accounting-clean sample of 1,232 firms will serve as a sensitivity sample rather than replacing the primary sample without further justification.

The statistical analysis will compare linear and nonlinear regression approaches using consistent sample definitions and out-of-sample evaluation. Transformations, robust estimation, and potential exclusions will be documented separately from the SEC data-engineering pipeline.


## 7. Conclusion and Next Steps

The SEC EDGAR data-engineering stage is complete for fiscal year 2024.

The extraction identified 1,467 eligible U.S. manufacturing firms, of which 1,285 (87.6%) have usable leverage and ROA observations. The resulting datasets preserve financial statement values, liabilities provenance, missingness indicators, and accounting review flags.

Three datasets are exported to `data/processed/`:

- `financials_full_2024.csv`: All eligible firms, including incomplete observations and diagnostic fields.
- `financials_primary_2024.csv`: The 1,285 observations with usable leverage and ROA.
- `financials_accounting_clean_2024.csv`: The 1,232 usable observations without accounting review flags.

A separate summary file, `sample_summary_2024.csv`, records sample counts and quality diagnostics.

### Next Notebook: `02_exploratory_analysis`

The next stage will be conducted in R and will focus on:

1. Distributional analysis of ROA and leverage.
2. Visualization of the leverage–profitability relationship.
3. Industry composition and subgroup differences.
4. Identification of influential observations.
5. Preparation of consistent samples for linear, spline, and GAM comparisons.

The Python pipeline is restricted to reproducible data acquisition, financial variable construction, and quality validation. Statistical transformations and modeling decisions will be implemented in R.
